# Analysis

Analysis of the final prediction and evaluation files by language and model.


## Load and summarize

Load the final prediction files and show only the dataset coverage: language, model, and number of evaluated questions.


In [31]:
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path.cwd() / "data"

files = [
    path for path in sorted(DATA_DIR.glob("prediction_*.csv"))
]

language_names = {"en": "English", "sw": "Swahili"}
frames = []
for path in files:
    parts = path.stem.split("_")
    language_code, model = parts[1], parts[2]
    frame = pd.read_csv(path).assign(
        language=language_names[language_code],
        model=model,
    )
    frames.append(frame)

predictions = pd.concat(frames, ignore_index=True)
answer_pattern = re.compile(r"<answer>\s*(?:[1-4])\s*</answer>")
predictions["format_error"] = ~predictions["raw_output"].fillna("").str.contains(answer_pattern)

input_price = {"small": 0.02, "big": 0.40}
output_price = {"small": 0.04, "big": 0.40}
predictions["input_cost"] = (
    predictions["prompt_tokens"] * predictions["model"].map(input_price) / 1_000_000
)
predictions["output_cost"] = (
    predictions["completion_tokens"] * predictions["model"].map(output_price) / 1_000_000
)

overview = predictions.groupby(["language", "model"], as_index=False).agg(
    n=("question_id", "count")
)
overview

,language,model,n
0,English,small,900
1,Swahili,small,900


## Model accuracy

Accuracy is reported with its sample standard deviation and a 95% Wilson confidence interval.


In [32]:
def wilson_interval(successes, trials, z=1.96):
    proportion = successes / trials
    denominator = 1 + z**2 / trials
    centre = (proportion + z**2 / (2 * trials)) / denominator
    margin = z * (proportion * (1 - proportion) / trials + z**2 / (4 * trials**2)) ** 0.5 / denominator
    return centre - margin, centre + margin

accuracy_rows = []
for (language, model), group in predictions.groupby(["language", "model"]):
    n = len(group)
    correct = int(group["correct"].sum())
    low, high = wilson_interval(correct, n)
    accuracy_rows.append({
        "language": language,
        "model": model,
        "n": n,
        "mean_accuracy": group["correct"].mean(),
        "sd_accuracy": group["correct"].std(ddof=1),
        "format_error_rate": group["format_error"].mean(),
        "95ci_accuracy": f"[{low:.1%}, {high:.1%}]",
    })

accuracy_table = pd.DataFrame(accuracy_rows)
display(accuracy_table.style.format({
    "mean_accuracy": "{:.1%}",
    "sd_accuracy": "{:.3f}",
    "format_error_rate": "{:.1%}",
}))

format_errors = predictions.loc[
    predictions["format_error"],
    ["language", "model", "question_id", "gold", "pred", "raw_output"],
].copy()
print(f"Format-error cases: {len(format_errors)}")
for _, case in format_errors.iterrows():
    print(f"{case['language']} / {case['model']} / {case['question_id']}")


,language,model,n,mean_accuracy,sd_accuracy,format_error_rate,95ci_accuracy
0,English,small,900,90.8%,0.290,0.4%,"[88.7%, 92.5%]"
1,Swahili,small,900,61.3%,0.487,1.3%,"[58.1%, 64.5%]"


Format-error cases: 16
English / small / q_149
English / small / q_215
English / small / q_680
English / small / q_857
Swahili / small / q_22
Swahili / small / q_156
Swahili / small / q_233
Swahili / small / q_286
Swahili / small / q_438
Swahili / small / q_510
Swahili / small / q_557
Swahili / small / q_620
Swahili / small / q_656
Swahili / small / q_734
Swahili / small / q_757
Swahili / small / q_864


## Token usage and inference cost

Token usage and monetary cost are shown in separate tables.


In [33]:
token_rows = []
for (language, model), group in predictions.groupby(["language", "model"]):
    total_tokens = group["prompt_tokens"] + group["completion_tokens"]
    token_rows.append({
        "language": language,
        "model": model,
        "n": len(group),
        "total_input_tokens": group["prompt_tokens"].sum(),
        "total_output_tokens": group["completion_tokens"].sum(),
        "total_tokens": total_tokens.sum(),
        "mean_in_tokens": group["prompt_tokens"].mean(),
        "sd_in_tokens": group["prompt_tokens"].std(ddof=1),
        "mean_out_tokens": group["completion_tokens"].mean(),
        "sd_out_tokens": group["completion_tokens"].std(ddof=1),
        "mean_total_tokens": total_tokens.mean(),
        "sd_total_tokens": total_tokens.std(ddof=1),
    })

token_table = pd.DataFrame(token_rows)
display( token_table.style.format({
    "total_input_tokens": "{:,.0f}",
    "total_output_tokens": "{:,.0f}",
    "total_tokens": "{:,.0f}",
    "mean_in_tokens": "{:.1f}",
    "sd_in_tokens": "{:.1f}",
    "mean_out_tokens": "{:.1f}",
    "sd_out_tokens": "{:.1f}",
    "mean_total_tokens": "{:.1f}",
    "sd_total_tokens": "{:.1f}",
}))

cost_rows = []
for (language, model), group in predictions.groupby(["language", "model"]):
    total_cost = group["input_cost"] + group["output_cost"]
    cost_rows.append({
        "language": language,
        "model": model,
        "n": len(group),
        "total_input_cost": group["input_cost"].sum(),
        "total_output_cost": group["output_cost"].sum(),
        "total_costs": total_cost.sum(),
        "mean_in_cost": group["input_cost"].mean(),
        "sd_in_cost": group["input_cost"].std(ddof=1),
        "mean_out_cost": group["output_cost"].mean(),
        "sd_out_cost": group["output_cost"].std(ddof=1),
        "mean_total_cost": (group["input_cost"] + group["output_cost"]).mean(),
        "sd_total_cost": (group["input_cost"] + group["output_cost"]).std(ddof=1),
    })

cost_table = pd.DataFrame(cost_rows)
display(cost_table.style.format({
    "total_input_cost": "${:,.6f}",
    "total_output_cost": "${:,.6f}",
    "total_costs": "${:,.6f}",
    "mean_in_cost": "${:.6f}",
    "sd_in_cost": "${:.6f}",
    "mean_out_cost": "${:.6f}",
    "sd_out_cost": "${:.6f}",
    "mean_total_cost": "${:.6f}",
    "sd_total_cost": "${:.6f}",
}))


,language,model,n,total_input_tokens,total_output_tokens,total_tokens,mean_in_tokens,sd_in_tokens,mean_out_tokens,sd_out_tokens,mean_total_tokens,sd_total_tokens
0,English,small,900,"209,446","133,678","343,124",232.7,36.1,148.5,72.0,381.2,86.8
1,Swahili,small,900,"355,142","10,372","365,514",394.6,71.7,11.5,32.4,406.1,80.3


,language,model,n,total_input_cost,total_output_cost,total_costs,mean_in_cost,sd_in_cost,mean_out_cost,sd_out_cost,mean_total_cost,sd_total_cost
0,English,small,900,$0.004189,$0.005347,$0.009536,$0.000005,$0.000001,$0.000006,$0.000003,$0.000011,$0.000003
1,Swahili,small,900,$0.007103,$0.000415,$0.007518,$0.000008,$0.000001,$0.000000,$0.000001,$0.000008,$0.000002
